# Organize Screenshots by App

Sort screenshot files into subfolders named after the app that produced them.

Filename pattern: `Screenshot_YYYYMMDD_HHMMSS_<AppName>.ext`

Examples:
- `Screenshot_20260329_094302_Reddit.png` -> `Reddit/`
- `Screenshot_20260201_181839_Mr D(1)(1).png` -> `Mr D/`
- `Screenshot_20260807_162248_Firefox(1).png` -> `Firefox/`

**Nothing moves until you explicitly uncomment and run the execute cell.**

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
src_dirs = [
    "packages/core/src",
    "packages/fs-indexer/src",
    "packages/metadata/src",
    "packages/media-grouping/src",
    "packages/duplicate-detection/src",
    "packages/rename-engine/src",
    "packages/job-runner/src",
    "packages/ui-common/src",
    "apps/media-curator/src",
    "apps/disk-atlas/src",
    "apps/rename-studio/src",
    "apps/control-center/src",
]
for d in src_dirs:
    p = str(ROOT / d)
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Project root: {ROOT}")

Project root: d:\Users\sousa\dev\repos\tidy-forge


## 1. Configuration

In [2]:
# ---- EDIT THESE ----
SOURCE_DIR = Path("F:/Media/Pictures/Screenshots")
DEST_DIR   = None   # None = in-place; or Path("F:/Media/Pictures/Screenshots-Sorted")
MODE       = "move" # "move" or "copy"

# Regex to extract the app name from screenshot filenames.
# First capture group = subfolder name.
APP_PATTERN = r"Screenshot_\d{8}_\d{6}_(.+?)(?:\(\d+\))*\.\w+$"
# --------------------

source = Path(SOURCE_DIR).resolve()
dest = Path(DEST_DIR).resolve() if DEST_DIR else source
print(f"Source:  {source}")
print(f"Dest:    {dest}")
print(f"Mode:    {MODE}")
print(f"Pattern: {APP_PATTERN}")

Source:  F:\Media\Pictures\Screenshots
Dest:    F:\Media\Pictures\Screenshots
Mode:    move
Pattern: Screenshot_\d{8}_\d{6}_(.+?)(?:\(\d+\))*\.\w+$


## 2. Scan Files

In [4]:
from tidyforge.fs_indexer import ExtensionFilter, scan_directory

# Screenshots are .jpg and .png
filters = [ExtensionFilter(include={".jpg", ".jpeg", ".png"})]
entries = list(scan_directory(source, filters=filters))
print(f"Found {len(entries):,} image files")

Found 18,665 image files


## 3. Preview: Group by App Name

See how many screenshots belong to each app before committing to anything.

In [5]:
from tidyforge.media_grouping import ByFilenamePattern, GroupingEngine

strategy = ByFilenamePattern(pattern=APP_PATTERN)
engine = GroupingEngine(strategy)
summaries = engine.group_summary(entries)

print(f"{len(summaries)} app groups:\n")
for s in sorted(summaries, key=lambda s: -s.count):
    print(f"  {s.key:30s}  {s.count:4d} files  {s.size_human:>8s}")

65 app groups:

  unmatched                       15809 files    4.6 GB
  Reddit                           620 files  405.5 MB
  Firefox                          561 files  321.6 MB
  eufy Life                        187 files   25.8 MB
  WhatsApp                         172 files  145.8 MB
  ChatGPT                          164 files   37.9 MB
  Duolingo                         114 files   21.7 MB
  OneDrive                          82 files  168.5 MB
  Garmin Connect                    80 files   22.2 MB
  LinkedIn                          74 files   65.0 MB
  YouTube Music                     62 files   95.9 MB
  Gmail                             61 files   28.2 MB
  Connect                           59 files    9.6 MB
  TikTok                            58 files   36.1 MB
  StandardStanbic Bank              48 files    9.6 MB
  YouTube                           47 files   34.0 MB
  Maps                              42 files   26.6 MB
  Huawei Health                     38 files    

## 4. Build the Organize Plan (Dry-Run)

Shows exactly where each file would go. **No files are moved yet.**

In [6]:
from media_curator.organize import build_organize_plan

plan = build_organize_plan(
    entries, dest, mode=MODE, group_fn=strategy.group_key,
)

# Validate
issues = plan.validate()
if issues:
    print(f"{len(issues)} issues:")
    for issue in issues[:20]:
        print(f"  {issue}")
    if len(issues) > 20:
        print(f"  ... and {len(issues) - 20} more")
else:
    print("No validation issues.")

print(f"\nPlan: {len(plan.actions)} files to {MODE}")

5340 issues:
  Collision: 2 files target 'F:\Media\Pictures\Screenshots\WhatsApp\Screenshot_20240527_222717_WhatsApp.png'
  Collision: 2 files target 'F:\Media\Pictures\Screenshots\unmatched\SmartSelect_20240519_072214_Firefox.png'
  Collision: 2 files target 'F:\Media\Pictures\Screenshots\unmatched\SmartSelect_20240520_064216_YouTube.png'
  Collision: 2 files target 'F:\Media\Pictures\Screenshots\unmatched\SmartSelect_20240522_022505_Reddit.png'
  Collision: 2 files target 'F:\Media\Pictures\Screenshots\unmatched\SmartSelect_20240524_102119_Firefox.png'
  Collision: 2 files target 'F:\Media\Pictures\Screenshots\unmatched\SmartSelect_20240524_125734_Maps.png'
  Collision: 2 files target 'F:\Media\Pictures\Screenshots\unmatched\SmartSelect_20240524_130008_Maps.png'
  Collision: 2 files target 'F:\Media\Pictures\Screenshots\unmatched\SmartSelect_20240524_130112_Maps.png'
  Collision: 2 files target 'F:\Media\Pictures\Screenshots\unmatched\SmartSelect_20240524_135536_WhatsApp.png'
  Colli

In [7]:
# Preview grouped by destination subfolder
from collections import defaultdict

groups = defaultdict(list)
for action in plan.actions:
    subfolder = action.destination.parent.name
    groups[subfolder].append(action)

for subfolder in sorted(groups):
    actions = groups[subfolder]
    print(f"\n{subfolder}/  ({len(actions)} files)")
    for a in actions[:3]:
        print(f"  {a.source.name}")
    if len(actions) > 3:
        print(f"  ... and {len(actions) - 3} more")


Bolt/  (2 files)
  Screenshot_20251207_143016_Bolt.png
  Screenshot_20251207_143016_Bolt.png

Bookingcom/  (10 files)
  Screenshot_20241111_155442_Bookingcom.png
  Screenshot_20241111_155507_Bookingcom.png
  Screenshot_20241111_155442_Bookingcom.png
  ... and 7 more

Bookingcom_resized/  (1 files)
  Screenshot_20241206_124306_Bookingcom_resized.jpg

Bookingcom_resized /  (8 files)
  Screenshot_20241206_124306_Bookingcom_resized (1).jpg
  Screenshot_20241206_124306_Bookingcom_resized (2).jpg
  Screenshot_20241206_124306_Bookingcom_resized (3).jpg
  ... and 5 more

Built With Science/  (26 files)
  Screenshot_20251229_070822_Built With Science.png
  Screenshot_20251229_070832_Built With Science.png
  Screenshot_20251229_070858_Built With Science.png
  ... and 23 more

Calendar/  (7 files)
  Screenshot_20241022_210258_Calendar.png
  Screenshot_20241022_210258_Calendar.png
  Screenshot_20241201_205905_Calendar.png
  ... and 4 more

Call/  (2 files)
  Screenshot_20250117_170633_Call.png
  

In [8]:
# Dry-run simulation
dry_manifest = plan.execute(dry_run=True)
print(dry_manifest.summary)

[DRY RUN] 18665 operations: 18665 succeeded, 0 failed


## 5. Execute (for real)

**Only run the cell below when you are satisfied with the preview above.**

We rebuild a fresh plan because the dry-run consumed the previous one's action statuses.

In [ ]:
# Rebuild fresh plan
final_plan = build_organize_plan(
    entries, dest, mode=MODE, group_fn=strategy.group_key,
)

# Uncomment the lines below to actually move/copy the files:
manifest = final_plan.execute(dry_run=False)
print(manifest.summary)

In [ ]:
# Optional: save the manifest to a JSON log
manifest.to_json(ROOT / "data" / "tmp" / "organize_by_app_manifest.json")
print("Manifest saved.")